# Lab 13 — 로깅 성능 벤치마크 (배포 → 측정 → 삭제)

**가설(공식 문서 근거)**
- **C1 로깅 없음**: 기준선(최소 오버헤드). 감사 불가 → 엔터프라이즈 부적합.
- **C2 App Insights body 8KB**: "payload 로깅은 성능을 크게 저하"·"1,000 req/s 초과 시 throughput 40~50% 감소". body↑ 시 오버헤드↑, **8KB 에서 평탄화** 예상.
- **C3 App Insights body=0 + Event Hub**: `log-to-eventhub` 는 "샘플링 무관·전량 로깅·200KB". 부하에서 상대적으로 평탄 예상(단, 인라인 body-read 비용 존재).

**측정 격리** — 백엔드는 `return-response` mock 으로 제거(`BackendTime≈0`). 권위 지표는 서버측 `TotalTime − BackendTime`(네트워크·백엔드 배제). GatewayLogs 진단은 전 구성 상시 ON(상수 오버헤드 → delta 상쇄).

> ⚠️ 이 랩은 시간당 과금 리소스를 씁니다. `scripts/deploy-logbench.sh` 로 배포하고, 끝나면 `scripts/teardown-logbench.sh` 로 반드시 삭제하세요.


In [ ]:
# 셀 1: 환경 + 헬퍼 + 리소스 로드
import os, sys, json, time, subprocess, tempfile
from pathlib import Path
import requests

sys.path.insert(0, str(Path.cwd()))
import benchlib as bl

DRY_RUN = os.environ.get("DRY_RUN", "false").lower() == "true"

def load_env(path=".env.logbench"):
    env = {}; p = Path(path)
    if not p.exists(): p = Path("../../.env.logbench")
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line: continue
            k, v = line.split("=", 1); env[k.strip()] = v.strip().strip('"').strip("'")
    return env
env = load_env()

RESOURCE_GROUP = env.get("LOGBENCH_RG", "")
APIM_NAME      = env.get("LOGBENCH_APIM_NAME", "")
APIM_URL       = env.get("LOGBENCH_APIM_URL", "").rstrip("/")
LA_CUSTOMER_ID = env.get("LOGBENCH_LA_CUSTOMER_ID", "")
LOADTEST_NAME  = env.get("LOGBENCH_LOADTEST_NAME", "")
EH_NAMESPACE   = env.get("LOGBENCH_EH_NAMESPACE", "")
EH_NAME        = env.get("LOGBENCH_EH_NAME", "logbench")
BENCH_API_ID   = "bench"
BENCH_SUB_KEY  = env.get("LOGBENCH_SUB_KEY", "")  # 셀에서 자동 발급/조회

def az(args):
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode
def az_json(args):
    out, err, rc = az(args)
    if rc != 0 or not out: return None
    try: return json.loads(out)
    except json.JSONDecodeError: return out

SUBSCRIPTION_ID = env.get("AZURE_SUBSCRIPTION_ID") or (az_json(["account","show","--query","id","-o","json"]) or "")
ARM_API = "2024-06-01-preview"
ARM_BASE = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
            f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}")

def arm(method, path, body=None, api=ARM_API):
    url = f"{ARM_BASE}{path}?api-version={api}"
    args = ["rest","--method",method,"--url",url]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w",suffix=".json",delete=False); json.dump(body,tmp); tmp.close()
        args += ["--headers","Content-Type=application/json","--body",f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp: os.unlink(tmp.name)
    return out, err, rc

def query_la(kql):
    """Log Analytics 워크스페이스(customerId GUID)로 KQL 실행 → 행 딕셔너리 리스트 반환."""
    if not LA_CUSTOMER_ID:
        print("  ⚠️ LA_CUSTOMER_ID 미설정 → KQL 건너뜀"); return None
    out, err, rc = az(["monitor","log-analytics","query","-w",LA_CUSTOMER_ID,
                       "--analytics-query",kql,"-o","json"])
    if rc != 0 or not out:
        print(f"  ⚠️ KQL 실패: {err[:160]}"); return None
    try: return json.loads(out)   # list[dict]: 각 행이 컬럼→값
    except json.JSONDecodeError: return None

def wait_propagation(sec=150):
    print(f"  ⏳ 설정 전파 대기 {sec}초 …"); time.sleep(sec); print("  ✅ 전파 대기 완료")

if DRY_RUN:
    print("🧪 DRY_RUN — Azure 호출 없이 흐름만 검증합니다.")
else:
    assert SUBSCRIPTION_ID and APIM_NAME and RESOURCE_GROUP, ".env.logbench / az login 을 확인하세요."
print("✅ 설정 완료 | APIM:", APIM_NAME or "(dry)", "| RG:", RESOURCE_GROUP or "(dry)")


In [ ]:
# 셀 2: bench API + echo operation + 구독키 프로비저닝
BENCH_PATH = "bench"

def provision_bench():
    if DRY_RUN:
        print("🧪 DRY_RUN: bench 프로비저닝 건너뜀"); return
    # API 생성
    arm("PUT", f"/apis/{BENCH_API_ID}", body={"properties": {
        "displayName": "Bench Echo", "path": BENCH_PATH, "protocols": ["https"],
        "subscriptionRequired": True}})
    # operation 생성 (GET /echo)
    arm("PUT", f"/apis/{BENCH_API_ID}/operations/echo", body={"properties": {
        "displayName": "echo", "method": "GET", "urlTemplate": "/echo"}})
    # 기본 정책(C1: log-to-eventhub 없음)
    arm("PUT", f"/apis/{BENCH_API_ID}/policies/policy",
        body={"properties": {"format": "rawxml", "value": bl.bench_policy_xml(False)}})
    # product 'bench-product' + API 연결 + 구독
    arm("PUT", "/products/bench-product", body={"properties": {
        "displayName": "Bench Product", "state": "published",
        "subscriptionRequired": True, "approvalRequired": False}})
    arm("PUT", f"/products/bench-product/apis/{BENCH_API_ID}", body={"properties": {}})
    arm("PUT", "/subscriptions/bench-sub", body={"properties": {
        "displayName": "bench-sub", "scope": f"/products/bench-product", "state": "active"}})
    print("  ✅ bench API/operation/product/subscription 생성")

def get_bench_key():
    global BENCH_SUB_KEY
    if DRY_RUN: BENCH_SUB_KEY = "dry-key"; return BENCH_SUB_KEY
    out, err, rc = az(["rest","--method","POST","--url",
        f"{ARM_BASE}/subscriptions/bench-sub/listSecrets?api-version={ARM_API}",
        "--query","primaryKey","-o","tsv"])
    BENCH_SUB_KEY = out.strip() if rc == 0 else ""
    print("  🔑 구독키 확보" if BENCH_SUB_KEY else f"  ⚠️ 키 확보 실패: {err[:120]}")
    return BENCH_SUB_KEY

provision_bench()
get_bench_key()

def call_bench(bytes_n, timeout=30):
    """bench echo 를 호출하고 (status, elapsed_ms) 반환."""
    n = bl.clamp_body_bytes(int(bytes_n))
    if DRY_RUN:
        return 200, 1.0
    t0 = time.perf_counter()
    r = requests.get(f"{APIM_URL}/{BENCH_PATH}/echo",
                     headers={"Ocp-Apim-Subscription-Key": BENCH_SUB_KEY},
                     params={"bytes": n}, timeout=timeout)
    return r.status_code, (time.perf_counter() - t0) * 1000.0


In [ ]:
# 셀 3: 구성 스위처 (C1/C2/C3) — API 정책 + applicationinsights 진단 토글
AI_LOGGER_ID = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/loggers/appinsights-logger"

def _set_ai_diagnostic(body_bytes):
    """None 이면 진단 삭제(C1), 아니면 응답 body 로깅 bytes 로 설정."""
    path = f"/apis/{BENCH_API_ID}/diagnostics/applicationinsights"
    if body_bytes is None:
        arm("DELETE", path); return
    arm("PUT", path, body={"properties": {
        "loggerId": AI_LOGGER_ID, "verbosity": "information",
        "sampling": {"samplingType": "fixed", "percentage": 100.0},
        "logClientIp": True, "httpCorrelationProtocol": "W3C",
        "frontend": {"response": {"body": {"bytes": body_bytes}}},
        "backend":  {"response": {"body": {"bytes": body_bytes}}}}})

def set_config(config):
    """config ∈ {'C1','C2','C3'} 적용: 정책(EH 유무) + 진단(body bytes). 전파 대기 포함."""
    assert config in ("C1", "C2", "C3")
    if DRY_RUN:
        print(f"🧪 DRY_RUN: set_config({config}) 스킵"); return
    xml = bl.bench_policy_xml(bl.config_uses_eventhub(config))
    arm("PUT", f"/apis/{BENCH_API_ID}/policies/policy",
        body={"properties": {"format": "rawxml", "value": xml}})
    _set_ai_diagnostic(bl.diagnostic_body_bytes(config))
    print(f"  ✅ {config} 적용 (EH={bl.config_uses_eventhub(config)}, "
          f"body_bytes={bl.diagnostic_body_bytes(config)})")
    wait_propagation(150)

print("구성 스위처 준비 완료: set_config('C1'|'C2'|'C3')")


## 1차 — body 크기 축 (저동시성 순차 측정)

body 크기를 1KB→200KB 로 키우며 구성별 레이턴시를 측정한다. 클라이언트 wall-clock 은 교차검증용이고, **권위 지표는 서버측 `TotalTime − BackendTime`**(다음 KQL 셀). 가설: C2 는 8KB 에서 평탄화, C3 는 크기에 따라 인라인 비용 증가.

In [ ]:
# 셀 4: 1차 러너 — 구성 × body 크기 스윕 (클라이언트 레이턴시)
BODY_SIZES = [1024, 4096, 8192, 16384, 65536, 204800]  # 1KB..200KB
WARMUP = 3
SAMPLES = 20

results_size = {}
for config in ["C1", "C2", "C3"]:
    set_config(config)
    per_size = []
    for n in BODY_SIZES:
        for _ in range(WARMUP):
            call_bench(n)
        lat = []
        for _ in range(SAMPLES):
            code, ms = call_bench(n)
            if code == 200: lat.append(ms)
        s = bl.summarize_latencies(lat)
        per_size.append({"bytes": n, **s})
        print(f"  {config} {n:>7}B  p50={s['p50']:.1f}ms p95={s['p95']:.1f}ms n={s['n']}")
    results_size[config] = per_size

print("✅ 1차 클라이언트 측정 완료")

In [ ]:
# 셀 5: 1차 서버측 게이트웨이 시간 (TotalTime - BackendTime) — 권위 지표
# GatewayLogs 수집 지연(~수 분) 가능 → 결과 0건이면 잠시 후 재실행.
kql = bl.gatewaylogs_kql(BENCH_API_ID, lookback_min=30)
rows = query_la(kql)
if rows:
    cols = list(rows[0].keys())
    print("  " + "  ".join(cols))
    for row in rows:
        print("  " + "  ".join(str(row.get(c)) for c in cols))
else:
    print("  (GatewayLogs 아직 없음 — 2~5분 후 이 셀을 다시 실행하세요)")

In [ ]:
# 셀 6: 1차 차트 — body 크기 vs 클라이언트 레이턴시(p95), 구성별 라인
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for config, rows in results_size.items():
    xs = [r["bytes"] / 1024 for r in rows]
    ys = [r["p95"] for r in rows]
    ax.plot(xs, ys, marker="o", label=config)
ax.set_xlabel("응답 body 크기 (KB)")
ax.set_ylabel("클라이언트 p95 레이턴시 (ms)")
ax.set_title("1차: body 크기별 레이턴시 (구성 비교)")
ax.axvline(8, color="gray", ls="--", alpha=0.6)  # App Insights 8KB cap
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()